# Imports

In [ ]:
import numpy as np

from molmod.constants import boltzmann, planck
from molmod.units import kjmol, angstrom, kcalmol, bar, kelvin, pascal, amu
from molmod.periodic import periodic as pp

from cmmdft.system import NanoporousHost, SphericalLJGuest, DualModelGuest
from cmmdft.extpot_calculator import read_pars_file, Interpolator, get_external_potential
from cmmdft.program import Program
from cmmdft.solver import Picard, Anderson, QuasiNewton
from cmmdft.eos import ModifiedBenedictWebbRubinEOS
from cmmdft.calculator import Calculator
from cmmdft.plotter import Plotter

from yaff import log as ylog
ylog.set_level(ylog.silent)

# System specifications

In [ ]:
# Host
hostname = 'ZIF8'
hostchk = '../InputFiles/Host/ZIF8/struct.chk'
hostpar = '../InputFiles/Host/ZIF8/pars_vdw_uff.txt'
host = NanoporousHost(name=hostname, chk=hostchk, par=hostpar)

guestname = 'CH4_UA'
sigma, epsilon = 3.75*angstrom, 0.29411*kcalmol
mass = 16.04*amu
guest = SphericalLJGuest(name=guestname, mass=mass, sigma=sigma, epsilon=epsilon)
eos = ModifiedBenedictWebbRubinEOS(mass, sigma, epsilon)


ff_suffix = 'UFF_TRAPPE' # force field names
funct_suffix = 'MFMT_MFA_WDA_c' # functional names
cutoff = 12.0*angstrom

## GRID SPECIFICATIONS
spacing = 0.5*angstrom
grid_suffix = '0_5' 

# Program setup

In [ ]:
temperature = 300.0*kelvin

program = Program(prefix=f'OutputFiles/', hostname=hostname, guestname=guestname, ff_suffix=ff_suffix, funct_suffix=funct_suffix,
                    grid_suffix=grid_suffix, overwrite=False)


program.set_system(host, guest)
program.set_grid(spacing=spacing)

# initialize free energy functionals
program.init_free_energy(temperature)
program.fener.add_hard_sphere(version='atWBII')
program.fener.add_mean_field(tailcorrections=True, cutoff=cutoff)
program.fener.add_correlation_wda_lj()
program.fener.add_external_potential(temperature=300, cutoff=cutoff)

program.fener.set_temperature(temperature)

# Setting the simulation condition

In [ ]:
pressures = np.linspace(1e-4,65,15)*bar
chempots = np.array([eos.calculate_mu(temperature,P) for P in pressures])
rho_bulk = eos.solve_densities_from_chempots(chempots)
rho_bulk = np.nanmin(rho_bulk, axis=1) # select lowest density solution

# Defining solver objects

In [ ]:
#defining multiple convergence criteria
criteria = ['riue', 'res']
thresholds = [1e-6, 1e-6]

# setup solvers for cascade
solvers = [Anderson(program, criterion=criteria, threshold=thresholds, track_history=True),
           QuasiNewton(program, method='bfgs', criterion=criteria, threshold=thresholds),
           Picard(program, alpha_mix=0.2, criterion=criteria, threshold=thresholds)]

# Running the cDFT simulation

In [ ]:
for e, mu in enumerate(chempots):
    program.cascade_solver(solvers=solvers, chempot=mu, Ninit=rho_bulk[e], rewrite=True)

# Saving the calculated loadings

In [ ]:
calc = Calculator(program)

#save loadings in .csv file
calc.save_loadings(300, chempots = chempots)
calc.save_loadings(300, chempots = chempots, pressure=True, eos=eos)
calc.save_loadings_AIF(300, chempots, eos=eos, loading_unit='mol/kg')